In [824]:
#Install packages
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import statsmodels.api as sm   # only for the Q-Q plot in the helper below

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.impute import SimpleImputer, KNNImputer

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

In [825]:
#Define Functions
def diagnostic_plots(y_true, y_pred):
    """The four residual plots from lecture, side by side."""
    resid = y_true - y_pred

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))

    axes[0].scatter(y_pred, resid, alpha=0.6)
    axes[0].axhline(0, color='red', linestyle='--')
    axes[0].set_xlabel('Fitted values')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title('Residuals vs. Fitted')

    axes[1].scatter(y_pred, y_true, alpha=0.6)
    lims = [min(y_pred.min(), y_true.min()), max(y_pred.max(), y_true.max())]
    axes[1].plot(lims, lims, color='red', linestyle='--')
    axes[1].set_xlabel('Fitted values')
    axes[1].set_ylabel('Actual values')
    axes[1].set_title('Actual vs. Fitted')

    sm.qqplot(resid, line='s', ax=axes[2])
    axes[2].set_title('Normal Q-Q')

    axes[3].hist(resid, bins=20, edgecolor='white')
    axes[3].axvline(0, color='red', linestyle='--')
    axes[3].set_xlabel('Residual')
    axes[3].set_title('Histogram of Residuals')

    plt.tight_layout()
    plt.show()


In [826]:
#Read and split data
df = pd.read_csv('Taxi-cancellation-case.csv',index_col=0)

X = df.drop(columns=['Car_Cancellation'])
y = df['Car_Cancellation']

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    stratify=y,
    random_state=42
)

y_train_cancellation_rate = y_train.sum()/len(y_train)
y_test_cancellation_rate = y_test.sum()/len(y_test)
print(y_train_cancellation_rate)
print(y_test_cancellation_rate)

X_train.shape

0.07425
0.0745


(8000, 17)

#### Question 1: Split data and benchmark cancellation rate
We split the data into a 80/20 train/test split using stratified sampling to ensure there was a representative number of cancellations in both our train and test set. We confirmed this by taking total cancellations divided by observations for both the train and test set. A cancellation rate of 7.4% for both datasets gives us comfort there was no unintentional drift.

In [827]:
#Assess Missingness
missingness_rate = (X_train.isna().sum() / len(X_train))
print(missingness_rate.sort_values(ascending=False))

to_city_id             0.966875
package_id             0.826375
from_city_id           0.627625
to_date                0.418250
to_long                0.206750
to_lat                 0.206750
to_area_id             0.206750
from_area_id           0.001625
from_long              0.001625
from_lat               0.001625
booking_created        0.000000
user_id                0.000000
mobile_site_booking    0.000000
online_booking         0.000000
vehicle_model_id       0.000000
travel_type_id         0.000000
from_date              0.000000
dtype: float64


In [828]:
#Analytics on the travel type id

print(X_train['travel_type_id'].value_counts())

null_travel_types = X_train.drop(columns='travel_type_id').isna().groupby(X_train['travel_type_id']).sum()
print(null_travel_types)

travel_type_id
2    6346
3    1389
1     265
Name: count, dtype: int64
                user_id  vehicle_model_id  package_id  from_area_id  \
travel_type_id                                                        
1                     0                 0         265            13   
2                     0                 0        6346             0   
3                     0                 0           0             0   

                to_area_id  from_city_id  to_city_id  from_date  to_date  \
travel_type_id                                                             
1                      265            40           0          0       12   
2                        0          3668        6346          0     2828   
3                     1389          1313        1389          0      506   

                online_booking  mobile_site_booking  booking_created  \
travel_type_id                                                         
1                            0                  

#### Question 2: Assess Missingness
travel type has an imapct on null values. The first observarion is only category 3 (hourly rentals) tend to use a package id. Additionally it seems most hourly rentals or long distance do not populate a destination area/city/lat/long. Interestingly it seems most point-to-point bookings only populate an area or lat/long or area data, leaving city id null. It seems the city identification is primarily used for long distance trips as this is the only location variable populated for this grouping.

| Structural Missingness| Real Gap |
|-----------------------|-----------|
|package_id|from_lat|
|to_area_id|from_long|
|to_city_id|to_date|
|to_lat|from_city_id|
|to_long|from_area_id|

In [829]:
y = X_train['to_date'].isna().astype(int)

X = pd.DataFrame({
    'online_booking': X_train['online_booking'],
    'mobile_site_booking': X_train['mobile_site_booking'],
    'to_lat_missing': X_train['to_lat'].isna().astype(int),
    'long_distance_1': (X_train['travel_type_id'] == 1).astype(int),
    'point_to_point_2': (X_train['travel_type_id'] == 2).astype(int),
    'hourly_rental_3': (X_train['travel_type_id'] == 3).astype(int),
    'from_city_id_missing': X_train['from_city_id'].isna().astype(int)

})

usable = X.notna().all(axis=1)
X, y = X[usable], y[usable]

print(len(X), "rides,", X.shape[1], "features")

8000 rides, 7 features


In [830]:
X_train_slice, X_test_slice, y_train_slice, y_test_slice = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train_slice.shape, X_test_slice.shape)

(6400, 7) (1600, 7)


In [831]:
# Majority-class basline (score to beat)

print("Share of rides missing to date:", round(y_train_slice.mean(), 3))
print("Majority-class baseline:            ", round(1 - y_train_slice.mean(), 3))

Share of rides missing to date: 0.418
Majority-class baseline:             0.582


In [832]:
# K-fold cross validation
# KNN Classifier: 11

folds = KFold(n_splits=10, shuffle=True, random_state=42)

for k in [1, 3, 5, 7, 11, 15, 21, 31]:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    scores = cross_val_score(pipe, X_train_slice, y_train_slice, cv=folds, scoring='accuracy')
    print("k =", k, " CV accuracy:", round(scores.mean(), 3))

k = 1  CV accuracy: 0.747
k = 3  CV accuracy: 0.835
k = 5  CV accuracy: 0.84
k = 7  CV accuracy: 0.841
k = 11  CV accuracy: 0.84
k = 15  CV accuracy: 0.841
k = 21  CV accuracy: 0.841
k = 31  CV accuracy: 0.841


In [833]:
# Pipeline creation

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=11))
knn.fit(X_train_slice, y_train_slice)

pred = knn.predict(X_test_slice)

In [834]:
print(confusion_matrix(y_test_slice, pred))

[[678 253]
 [ 19 650]]


In [835]:
print("Baseline (predict 'present' always):", round(1 - y_test_slice.mean(), 3))
print()
print("Accuracy: ", round(accuracy_score(y_test_slice, pred), 3))
print("Precision:", round(precision_score(y_test_slice, pred), 3))
print("Recall:   ", round(recall_score(y_test_slice, pred), 3))

Baseline (predict 'present' always): 0.582

Accuracy:  0.83
Precision: 0.72
Recall:    0.972


### Question 3
The classifier beats the baseline (.83 > .582), whether a ride has a to_date or not
is predictable from whether or not from_city_id is missing or not.
to_date is not missing completly at random.

In [836]:
X_train.head(10)

,user_id,vehicle_model_id,package_id,travel_type_id,from_area_id,to_area_id,from_city_id,to_city_id,from_date,to_date,online_booking,mobile_site_booking,booking_created,from_lat,from_long,to_lat,to_long
row#,,,,,,,,,,,,,,,,,
9818,39476,12,NaN,2,1237.0,393.0,15.0,NaN,8/15/2013 3:50,8/15/2013 5:03,0,0,8/14/2013 17:37,12.926450,77.612060,13.199560,77.706880
8710,41644,12,1.0,3,1026.0,NaN,15.0,NaN,11/5/2013 7:40,11/5/2013 10:20,0,0,10/31/2013 9:37,13.104820,77.576310,NaN,NaN
1405,25651,12,NaN,1,392.0,NaN,15.0,41.0,3/8/2013 6:28,3/8/2013 0:00,0,0,3/7/2013 13:42,12.986284,77.735245,NaN,NaN
4779,33804,54,NaN,1,571.0,NaN,1.0,32.0,7/13/2013 6:57,7/13/2013 0:00,1,0,7/8/2013 14:09,12.951850,77.696420,NaN,NaN
7861,30869,12,1.0,3,1017.0,NaN,NaN,NaN,10/5/2013 8:24,10/5/2013 13:52,0,0,10/4/2013 11:01,13.110840,77.600740,NaN,NaN
5948,2961,12,NaN,1,182.0,NaN,15.0,32.0,8/12/2013 10:04,8/12/2013 23:59,1,0,8/11/2013 18:29,12.945740,77.649280,NaN,NaN
1706,23109,12,NaN,2,393.0,1254.0,NaN,NaN,3/24/2013 18:14,3/25/2013 0:00,1,0,3/24/2013 9:26,13.199560,77.706880,12.934048,77.580568
106,21727,85,NaN,2,393.0,793.0,NaN,NaN,1/6/2013 21:36,NaN,0,0,1/6/2013 12:07,13.199560,77.706880,13.022390,77.594920
5114,33243,12,NaN,2,1395.0,363.0,NaN,NaN,7/19/2013 6:57,7/19/2013 8:15,0,0,7/18/2013 21:24,13.005603,77.628214,12.992870,77.716620


In [837]:
missingness_rate = (X_train.isna().sum() / len(X_train))
print(missingness_rate.sort_values(ascending=False))

to_city_id             0.966875
package_id             0.826375
from_city_id           0.627625
to_date                0.418250
to_long                0.206750
to_lat                 0.206750
to_area_id             0.206750
from_area_id           0.001625
from_long              0.001625
from_lat               0.001625
booking_created        0.000000
user_id                0.000000
mobile_site_booking    0.000000
online_booking         0.000000
vehicle_model_id       0.000000
travel_type_id         0.000000
from_date              0.000000
dtype: float64


In [838]:
X_train.columns

Index(['user_id', 'vehicle_model_id', 'package_id', 'travel_type_id',
       'from_area_id', 'to_area_id', 'from_city_id', 'to_city_id', 'from_date',
       'to_date', 'online_booking', 'mobile_site_booking', 'booking_created',
       'from_lat', 'from_long', 'to_lat', 'to_long'],
      dtype='str')

In [839]:
X_train.dtypes

user_id                  int64
vehicle_model_id         int64
package_id             float64
travel_type_id           int64
from_area_id           float64
to_area_id             float64
from_city_id           float64
to_city_id             float64
from_date                  str
to_date                    str
online_booking           int64
mobile_site_booking      int64
booking_created            str
from_lat               float64
from_long              float64
to_lat                 float64
to_long                float64
dtype: object

In [840]:
X_train.nunique()

user_id                5831
vehicle_model_id         20
package_id                6
travel_type_id            3
from_area_id            495
to_area_id              454
from_city_id              2
to_city_id               61
from_date              6645
to_date                4329
online_booking            2
mobile_site_booking       2
booking_created        7746
from_lat                401
from_long               398
to_lat                  372
to_long                 369
dtype: int64

In [841]:
X_train['package_id_missing'] = np.where(X_train['package_id'].isna(),1,0)
X_train['to_date_missing'] = np.where(X_train['to_date'].isna(),1,0)
X_train = X_train.drop(columns=['user_id','package_id','to_city_id','from_city_id','to_date'])

drop_rows = X_train[X_train['from_area_id'].isna() | X_train['from_lat'].isna() | X_train['from_long'].isna()].index
X_train = X_train.drop(index=drop_rows)
y_train = y_train.drop(index=drop_rows)

X_train = pd.get_dummies(X_train, columns=['vehicle_model_id','travel_type_id'],drop_first=True,dtype=int)

#Deal with the dates, used AI to help engineer numeric feautres from datetime data
for c in ['from_date', 'booking_created']:
    X_train[c] = pd.to_datetime(X_train[c])

X_train['booking_days_before_trip'] = (X_train['from_date'] - X_train['booking_created']).dt.total_seconds() / 86400
X_train['from_hour']  = X_train['from_date'].dt.hour
X_train['from_dow']   = X_train['from_date'].dt.dayofweek
X_train['from_month'] = X_train['from_date'].dt.month

X_train = X_train.drop(columns=['from_date', 'booking_created'])
X_train = pd.get_dummies(X_train, columns=['from_hour', 'from_dow', 'from_month'], drop_first=True,dtype=int)

X_train.shape

/var/folders/m9/_pg_9rkd0ngdmqrhjmr8pk6c0000gn/T/ipykernel_8634/1196849201.py:17: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  X_train['from_dow']   = X_train['from_date'].dt.dayofweek


(7987, 70)

In [842]:
drop_cols = ['to_lat']
features = [c for c in X_train.columns if c not in drop_cols]
X_train_area = X_train[features]

has_area_destination = X_train['to_lat'].notna() & X_train_area[features].notna().all(axis=1)

X_area = X_train.loc[has_area_destination, features]
y_area = X_train.loc[has_area_destination, 'to_lat']

print(len(X_area), "cars have an area destination with every feature present")
print("Lat range: mean", round(y_area.mean()), " std", round(y_area.std()))

6346 cars have an area destination with every feature present
Lat range: mean 13  std 0


In [843]:
folds = KFold(n_splits=10, shuffle=True, random_state=42)

for k in [10, 30, 50, 70, 110, 150, 210, 310, 510, 1010]:
    pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    scores = cross_val_score(pipe, X_area, y_area, cv=folds,
                             scoring='neg_root_mean_squared_error')
    print("k =", k, " CV RMSE: ", round(-scores.mean(),ndigits=4))

k = 10  CV RMSE:  0.0995
k = 30  CV RMSE:  0.0984
k = 50  CV RMSE:  0.0985
k = 70  CV RMSE:  0.0982
k = 110  CV RMSE:  0.0982
k = 150  CV RMSE:  0.0985
k = 210  CV RMSE:  0.0994
k = 310  CV RMSE:  0.1005
k = 510  CV RMSE:  0.1023
k = 1010  CV RMSE:  0.1052


In [844]:
X_train.dtypes

from_area_id                float64
to_area_id                  float64
online_booking                int64
mobile_site_booking           int64
from_lat                    float64
from_long                   float64
to_lat                      float64
to_long                     float64
package_id_missing            int64
to_date_missing               int64
vehicle_model_id_10           int64
vehicle_model_id_12           int64
vehicle_model_id_17           int64
vehicle_model_id_23           int64
vehicle_model_id_24           int64
vehicle_model_id_28           int64
vehicle_model_id_30           int64
vehicle_model_id_36           int64
vehicle_model_id_54           int64
vehicle_model_id_64           int64
vehicle_model_id_65           int64
vehicle_model_id_85           int64
vehicle_model_id_86           int64
vehicle_model_id_87           int64
vehicle_model_id_89           int64
vehicle_model_id_90           int64
vehicle_model_id_91           int64
travel_type_id_2            

In [845]:
prep = make_pipeline(StandardScaler(), KNNImputer(n_neighbors=70))
prep.fit(X_train)

Z_train = pd.DataFrame(prep.transform(X_train), columns=X_train.columns, index=X_train.index)
# Z_test = pd.DataFrame(prep.transform(X_test), columns=X.columns, index=X_test.index)

print("Gaps remaining:", int(Z_train.isna().sum().sum()), "train")
      # int(Z_test.isna().sum().sum()), "test")

Gaps remaining: 0 train


In [846]:
scaler = prep.named_steps['standardscaler']

filled_train = pd.DataFrame(scaler.inverse_transform(Z_train),
                            columns=X_train.columns, index=X_train.index)

check = pd.DataFrame({
    'observed_mean': X_train.mean().round(1),
    'after_imputation': filled_train.mean().round(1),
    'n_missing': X_train.isna().sum(),
})


check.loc[check['n_missing'] > 0]

,observed_mean,after_imputation,n_missing
to_area_id,662.6,666.3,1641
to_lat,13.0,13.0,1641
to_long,77.6,77.6,1641


In [847]:
missingness_rate = (X_train.isna().sum() / len(X_train))
print(missingness_rate.sort_values(ascending=False))

to_area_id                  0.205459
to_lat                      0.205459
to_long                     0.205459
from_hour_16                0.000000
from_hour_22                0.000000
from_hour_21                0.000000
from_hour_20                0.000000
from_hour_19                0.000000
from_hour_18                0.000000
from_hour_17                0.000000
from_hour_14                0.000000
from_hour_15                0.000000
from_dow_1                  0.000000
from_hour_13                0.000000
from_hour_12                0.000000
from_hour_11                0.000000
from_hour_10                0.000000
from_hour_9                 0.000000
from_hour_23                0.000000
from_area_id                0.000000
from_hour_8                 0.000000
from_dow_3                  0.000000
from_dow_4                  0.000000
from_dow_5                  0.000000
from_dow_6                  0.000000
from_month_2                0.000000
from_month_3                0.000000
f

In [848]:
#Drop columns & fill missing values
# X_train = X_train.drop(columns=['user_id']) #Drop user_id bcz lots of values, drop city id bcz high null rate
# X_train['package_id'] = X_train['package_id'].fillna(0)

# #Transform categorical variables to dummy variables
# categories = ['vehicle_model_id', 'package_id', 'travel_type_id']
# X_train[categories] = X_train[categories].astype('category')
# X_train = pd.get_dummies(X_train, columns=categories, drop_first=True,dtype=int)
# X_train.shape

# #Deal with the dates, used AI to help engineer numeric feautres from datetime data
# for c in ['from_date', 'to_date', 'booking_created']:
#     X_train[c] = pd.to_datetime(X_train[c])

# X_train['booking_days_before_trip'] = (X_train['from_date'] - X_train['booking_created']).dt.total_seconds() / 86400
# X_train['from_hour']  = X_train['from_date'].dt.hour
# X_train['from_dow']   = X_train['from_date'].dt.dayofweek
# X_train['from_month'] = X_train['from_date'].dt.month
# X_train['trip_hours'] = (X_train['to_date'] - X_train['from_date']).dt.total_seconds() / 3600   # NaN where to_date is missing

# X_train = X_train.drop(columns=['from_date', 'to_date', 'booking_created'])
# X_train = pd.get_dummies(X_train, columns=['from_hour', 'from_dow', 'from_month'], drop_first=True,dtype=int)

# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)
# X_train.head()


In [849]:
missingness_rate = (X_train.isna().sum() / len(X_train))
print(missingness_rate.sort_values(ascending=False))

to_area_id                  0.205459
to_lat                      0.205459
to_long                     0.205459
from_hour_16                0.000000
from_hour_22                0.000000
from_hour_21                0.000000
from_hour_20                0.000000
from_hour_19                0.000000
from_hour_18                0.000000
from_hour_17                0.000000
from_hour_14                0.000000
from_hour_15                0.000000
from_dow_1                  0.000000
from_hour_13                0.000000
from_hour_12                0.000000
from_hour_11                0.000000
from_hour_10                0.000000
from_hour_9                 0.000000
from_hour_23                0.000000
from_area_id                0.000000
from_hour_8                 0.000000
from_dow_3                  0.000000
from_dow_4                  0.000000
from_dow_5                  0.000000
from_dow_6                  0.000000
from_month_2                0.000000
from_month_3                0.000000
f

In [850]:
#Use a reasonable missing column to tune K
features = X_train.drop(columns=['to_lat']).columns
has_lat = X_train['to_lat'].notna() & X_train.notna().all(axis=1)

X_lat = X_train.loc[has_lat, features]
y_lat = X_train.loc[has_lat, 'to_lat']

print(len(X_lat), "rides report latitude and have every feature present")
print("Latitude: mean", round(y_lat.mean()), " std", round(y_lat.std()))

6346 rides report latitude and have every feature present
Latitude: mean 13  std 0


In [851]:
folds = KFold(n_splits=10, shuffle=True, random_state=42)

for k in [10, 30, 50, 70, 110, 150, 210, 310, 510, 1010]:
    pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    scores = cross_val_score(pipe, X_lat, y_lat, cv=folds,
                             scoring='neg_root_mean_squared_error')
    print("k =", k, " CV RMSE: $", round(-scores.mean()))

k = 10  CV RMSE: $ 0
k = 30  CV RMSE: $ 0
k = 50  CV RMSE: $ 0
k = 70  CV RMSE: $ 0
k = 110  CV RMSE: $ 0
k = 150  CV RMSE: $ 0
k = 210  CV RMSE: $ 0
k = 310  CV RMSE: $ 0
k = 510  CV RMSE: $ 0
k = 1010  CV RMSE: $ 0
